# All-in-One-Gait Final Clean Colab Notebook

This clean notebook runs our course-project version of gait gallery/probe recognition on Colab.

Credit: this project is built on top of the original All-in-One-Gait repository by Dongyang Jin and collaborators: https://github.com/jdyjjj/All-in-One-Gait. Our additions are the clean gallery/probe scripts, threshold/closed-set wrappers, annotation renderer, and Gradio interface.

## 1. Runtime Check

Use a GPU runtime in Colab: `Runtime > Change runtime type > T4/A100/L4 GPU`.

In [ ]:
import sys, platform, os
print('python:', sys.version)
print('platform:', platform.platform())
try:
    import torch
    print('torch:', torch.__version__)
    print('cuda available:', torch.cuda.is_available())
    print('gpu count:', torch.cuda.device_count())
    if torch.cuda.is_available():
        print('gpu name:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('torch check failed:', repr(exc))


## 2. Clone The Project Repo

The repo below is our course-project fork/bundle. It contains the original All-in-One-Gait code plus our tracked helper scripts. The notebook also writes the final clean-demo scripts explicitly, so it is safe on a fresh Colab runtime.

In [ ]:
%cd /content
REPO_URL = 'https://github.com/PritomKumarPaul/gait_tracking.git'
REPO_DIR = '/content/All-in-One-Gait'

import os
from pathlib import Path
if not Path(REPO_DIR).exists():
    !git clone --depth 1 $REPO_URL $REPO_DIR
else:
    print('Repo already exists:', REPO_DIR)
%cd /content/All-in-One-Gait
!git status --short


## 3. Install Dependencies

Important Colab note: current Colab often starts with NumPy 2.x. Some OpenCV/Paddle wheels used by this older project expect the NumPy 1.x ABI, so this cell pins `numpy<2`. If you already imported `cv2` before running this, restart the runtime after this cell and continue from the top.

In [ ]:
%cd /content/All-in-One-Gait

# Hard constraint: this project path needs NumPy 1.x for OpenCV/cython-bbox ABI compatibility.
!printf 'numpy<2
' > /content/allinonegait_constraints.txt

# Remove Colab/OpenCV wheels that may force NumPy 2.x.
!python -m pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless cython-bbox

!PIP_CONSTRAINT=/content/allinonegait_constraints.txt python -m pip install -q --upgrade pip setuptools wheel
!PIP_CONSTRAINT=/content/allinonegait_constraints.txt python -m pip install -q --force-reinstall "numpy<2" "Cython<3"

# Install dependencies under the NumPy<2 constraint.
!PIP_CONSTRAINT=/content/allinonegait_constraints.txt python -m pip install -q --no-deps "opencv-python-headless>=4.8,<4.11"
!PIP_CONSTRAINT=/content/allinonegait_constraints.txt python -m pip install -q filelock filterpy h5py kornia lap loguru motmetrics ninja prettytable pyyaml scikit-image scikit-learn scipy tabulate tensorboard thop tqdm visualdl gdown gradio imageio-ffmpeg paddlepaddle

# Force NumPy back below 2, then build cython-bbox last against that exact runtime NumPy.
!PIP_CONSTRAINT=/content/allinonegait_constraints.txt python -m pip install -q --force-reinstall "numpy<2" "Cython<3"
!PIP_CONSTRAINT=/content/allinonegait_constraints.txt python -m pip install -q --no-build-isolation --no-cache-dir cython-bbox==0.1.3

# IMPORTANT: if this cell changed NumPy in an already-running notebook, restart runtime once
# before running the verification/demo cells. On a fresh runtime, the check below should pass.
import numpy as np
import cv2
print('numpy:', np.__version__)
print('cv2:', cv2.__version__)
import cython_bbox
import loguru, thop, lap
print('cython_bbox/loguru/thop/lap: ok')
try:
    import paddle
    print('paddle:', paddle.__version__)
except Exception as exc:
    print('paddle import failed:', repr(exc))


## 4. Patch Small Compatibility Issues

This keeps the notebook robust across original/upstream and our forked copy. The patches are safe if they are already present.

In [ ]:
from pathlib import Path
ROOT = Path('/content/All-in-One-Gait')

# Some Paddle versions moved `core` from paddle.fluid to paddle.base.
infer_py = ROOT / 'OpenGait/demo/libs/paddle/infer.py'
if infer_py.exists():
    text = infer_py.read_text()
    old = 'import paddle.fluid.core as core'
    if old in text:
        text = text.replace(old, 'try:
    import paddle.fluid.core as core
except Exception:
    from paddle.base import core')
        infer_py.write_text(text)
    print('patched paddle infer:', infer_py)
else:
    print('missing infer.py:', infer_py)


## 5. Download Required Checkpoints

This downloads three things: ByteTrack person tracker, Paddle human segmentation model, and GREW GaitBase weights. GREW GaitGL is optional and can be enabled for comparison.

In [ ]:
from pathlib import Path
ROOT = Path('/content/All-in-One-Gait')
OPENGAIT = ROOT / 'OpenGait'
%cd /content/All-in-One-Gait/OpenGait

!mkdir -p demo/checkpoints/bytetrack_model demo/checkpoints/seg_model demo/checkpoints/gait_model demo/output output

# ByteTrack detector checkpoint. The exp file is tracked in the repo; the .pth.tar is too large, so download it.
BYTETRACK_CKPT = OPENGAIT / 'demo/checkpoints/bytetrack_model/bytetrack_x_mot17.pth.tar'
if not BYTETRACK_CKPT.exists():
    !gdown https://drive.google.com/uc?id=1P4mY0Yyd3PPTybgZkjMYhFri88nTmJX5 -O demo/checkpoints/bytetrack_model/bytetrack_x_mot17.pth.tar
print('ByteTrack checkpoint:', BYTETRACK_CKPT.exists(), BYTETRACK_CKPT)

# Human segmentation model used by the original demo pipeline.
SEG_DIR = OPENGAIT / 'demo/checkpoints/seg_model/human_pp_humansegv2_mobile_192x192_inference_model_with_softmax'
if not (SEG_DIR / 'deploy.yaml').exists():
    !wget -nc -O demo/checkpoints/seg_model/human_pp_humansegv2_mobile_192x192_inference_model_with_softmax.zip https://paddleseg.bj.bcebos.com/dygraph/pp_humanseg_v2/human_pp_humansegv2_mobile_192x192_inference_model_with_softmax.zip
    !unzip -n demo/checkpoints/seg_model/human_pp_humansegv2_mobile_192x192_inference_model_with_softmax.zip -d demo/checkpoints/seg_model/
print('Segmentation model:', (SEG_DIR / 'deploy.yaml').exists(), SEG_DIR)

# GREW GaitBase, recommended default.
GAITBASE_CKPT = OPENGAIT / 'demo/checkpoints/gait_model/GREW/Baseline/GaitBase_DA/checkpoints/GaitBase_DA-180000.pt'
if not GAITBASE_CKPT.exists():
    !wget -nc -O demo/checkpoints/gait_model/pretrained_grew_gaitbase.zip https://github.com/ShiqiYu/OpenGait/releases/download/v2.0/pretrained_grew_gaitbase.zip
    !unzip -n demo/checkpoints/gait_model/pretrained_grew_gaitbase.zip -d demo/checkpoints/gait_model/
print('GREW GaitBase:', GAITBASE_CKPT.exists(), GAITBASE_CKPT)

# Optional GREW GaitGL. Leave DOWNLOAD_GAITGL=False unless you want the larger ablation model.
DOWNLOAD_GAITGL = False
GAITGL_CKPT = OPENGAIT / 'demo/checkpoints/gait_model/GaitGL/checkpoints/GaitGL-250000.pt'
if DOWNLOAD_GAITGL and not GAITGL_CKPT.exists():
    !wget -nc -O demo/checkpoints/gait_model/pretrained_grew_gaitgl.zip https://github.com/ShiqiYu/OpenGait/releases/download/v1.1/pretrained_grew_gaitgl.zip
    !unzip -n demo/checkpoints/gait_model/pretrained_grew_gaitgl.zip -d demo/checkpoints/gait_model/
print('GREW GaitGL optional:', GAITGL_CKPT.exists(), GAITGL_CKPT)


## 6. Write Our Clean Demo Tools

This cell makes the notebook self-contained. It writes the helper tools into `OpenGait/tools/` and `clean_demo_v2/`, even if those files were not present in the cloned repo.

In [ ]:
from pathlib import Path
ROOT = Path('/content/All-in-One-Gait')
path = ROOT / 'OpenGait/tools/probe_only_entry_analysis.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('import argparse\nimport json\nimport os\nimport sys\nfrom pathlib import Path\nfrom typing import Dict, List, Optional\n\nimport numpy as np\nimport torch\n\n\nREPO_ROOT = Path(__file__).resolve().parents[1]\nPROJECT_ROOT = REPO_ROOT.parent\n\nos.chdir(REPO_ROOT)\n\nsys.path.append(str(REPO_ROOT / "demo" / "libs"))\nsys.path.append(str(REPO_ROOT))\nsys.path.append(str(REPO_ROOT / "opengait"))\n\nfrom demo.libs.track import track  # noqa: E402\nfrom demo.libs.segment import seg  # noqa: E402\nimport demo.libs.model.baselineDemo as baseline_demo  # noqa: E402\nfrom opengait.utils import config_loader  # noqa: E402\nfrom opengait.modeling import models  # noqa: E402\n\n\ndef log(message: str) -> None:\n    print(message, flush=True)\n\n\ndef cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:\n    a = a.astype(np.float32).reshape(-1)\n    b = b.astype(np.float32).reshape(-1)\n    a_norm = np.linalg.norm(a)\n    b_norm = np.linalg.norm(b)\n    if a_norm == 0.0 or b_norm == 0.0:\n        return -1.0\n    return float(np.dot(a, b) / (a_norm * b_norm))\n\n\ndef get_track_id(inputs) -> str:\n    return str(inputs[2][0])\n\n\ndef get_sequence_name(inputs) -> str:\n    return str(inputs[3][0])\n\n\ndef get_num_frames(inputs) -> int:\n    seq_len = inputs[4]\n    if seq_len is None:\n        return 0\n    return int(seq_len[0][0])\n\n\ndef build_opengait_model(cfgs: Dict, checkpoint_path: Path):\n    model_name = cfgs["model_cfg"]["model"]\n    model_cls = getattr(models, model_name)\n    model = model_cls.__new__(model_cls)\n    torch.nn.Module.__init__(model)\n    model.cfgs = cfgs\n    model.engine_cfg = cfgs["evaluator_cfg"]\n    model.iteration = 0\n    model.msg_mgr = None\n    model.build_network(cfgs["model_cfg"])\n    checkpoint = torch.load(str(checkpoint_path), map_location=torch.device("cuda" if torch.cuda.is_available() else "cpu"))\n    model.load_state_dict(checkpoint["model"], strict=cfgs["evaluator_cfg"]["restore_ckpt_strict"])\n    model.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))\n    model.eval()\n    model._expects_numeric_labels_for_inference = True\n    return model\n\n\ndef build_gaitbase_demo_model(cfg_path: Path, checkpoint_path: Path):\n    baseline_demo.model_cfgs["gait_model"] = str(checkpoint_path)\n    model = baseline_demo.BaselineDemo(config_loader(str(cfg_path)), training=False)\n    model.requires_grad_(False)\n    model.eval()\n    model._expects_numeric_labels_for_inference = False\n    return model\n\n\nMODEL_PROFILES = {\n    "grew_gaitbase": {\n        "display_name": "GREW GaitBase",\n        "loader": "baseline_demo",\n        "config": REPO_ROOT / "configs" / "gaitbase" / "gaitbase_da_gait3d.yaml",\n        "checkpoint": REPO_ROOT / "demo" / "checkpoints" / "gait_model" / "GREW" / "Baseline" / "GaitBase_DA" / "checkpoints" / "GaitBase_DA-180000.pt",\n        "archive_hint": REPO_ROOT / "demo" / "checkpoints" / "gait_model" / "pretrained_grew_gaitbase.zip",\n        "dataset_name": "GREW",\n    },\n    "current_gaitbase": {\n        "display_name": "Current Local GaitBase",\n        "loader": "baseline_demo",\n        "config": REPO_ROOT / "configs" / "gaitbase" / "gaitbase_da_gait3d.yaml",\n        "checkpoint": REPO_ROOT / "demo" / "checkpoints" / "gait_model" / "GaitBase_DA-180000.pt",\n        "dataset_name": "Gait3D",\n    },\n    "grew_gaitgl": {\n        "display_name": "GREW GaitGL",\n        "loader": "gaitgl",\n        "config": REPO_ROOT / "configs" / "gaitgl" / "gaitgl_GREW.yaml",\n        "checkpoint": REPO_ROOT / "demo" / "checkpoints" / "gait_model" / "GaitGL" / "checkpoints" / "GaitGL-250000.pt",\n        "archive_hint": REPO_ROOT / "demo" / "checkpoints" / "gait_model" / "pretrained_grew_gaitgl.zip",\n        "dataset_name": "GREW",\n    },\n}\n\n\ndef build_model(model_name: str):\n    profile = MODEL_PROFILES[model_name]\n    checkpoint_path = profile["checkpoint"]\n    log(f"[model] preparing profile={model_name} ({profile[\'display_name\']})")\n    log(f"[model] checkpoint={checkpoint_path}")\n    if not checkpoint_path.exists():\n        archive_hint = profile.get("archive_hint")\n        hint = f" Expected checkpoint: {checkpoint_path}."\n        if archive_hint is not None:\n            hint += f" If you downloaded the release asset, make sure {archive_hint.name} is extracted under demo/checkpoints/gait_model."\n        raise FileNotFoundError(\n            f"Selected model \'{model_name}\' is not ready locally.{hint}"\n        )\n\n    if profile["loader"] == "baseline_demo":\n        log("[model] using BaselineDemo loader for GaitBase-compatible checkpoint")\n        model = build_gaitbase_demo_model(profile["config"], checkpoint_path)\n        log(f"[model] ready type={type(model).__name__}")\n        return model, profile\n\n    cfgs = config_loader(str(profile["config"]))\n    cfgs["data_cfg"]["dataset_name"] = profile["dataset_name"]\n    cfgs["data_cfg"]["test_dataset_name"] = profile["dataset_name"]\n    cfgs["evaluator_cfg"]["enable_float16"] = False\n    cfgs["evaluator_cfg"]["restore_ckpt_strict"] = True\n    cfgs["evaluator_cfg"]["transform"] = [{"type": "BaseSilTransform"}]\n\n    log(f"[model] using OpenGait loader for model={cfgs[\'model_cfg\'][\'model\']}")\n    model = build_opengait_model(cfgs, checkpoint_path)\n    model.requires_grad_(False)\n    model.eval()\n    log(f"[model] ready type={type(model).__name__}")\n    return model, profile\n\n\ndef extract_embedding(model, inputs) -> np.ndarray:\n    with torch.no_grad():\n        if getattr(model, "_expects_numeric_labels_for_inference", False):\n            prepared_inputs = (inputs[0], [0], inputs[2], inputs[3], inputs[4])\n        else:\n            prepared_inputs = inputs\n        prepared = model.inputs_pretreament(prepared_inputs)\n        retval = model.forward(prepared)\n        if isinstance(retval, tuple):\n            retval = retval[0]\n        emb = retval["inference_feat"]["embeddings"].detach().cpu().numpy()[0]\n    return emb\n\n\ndef analyze_video(\n    model,\n    video_path: Path,\n    work_root: Path,\n    min_frames: int,\n) -> List[Dict]:\n    video_name = video_path.stem\n    video_output_dir = work_root / "tracking" / video_name\n    sil_root = work_root / "silhouettes"\n    video_output_dir.mkdir(parents=True, exist_ok=True)\n\n    log(f"[video:{video_name}] starting")\n    log(f"[video:{video_name}] tracking -> {video_output_dir}")\n    track_result = track(str(video_path), str(video_output_dir))\n    frame_hits = len(track_result)\n    track_ids = sorted({int(item[0]) for values in track_result.values() for item in values}) if track_result else []\n    log(f"[video:{video_name}] tracking done, frames_with_tracks={frame_hits}, unique_track_ids={track_ids}")\n\n    log(f"[video:{video_name}] segmentation -> {sil_root}")\n    sil_inputs = seg(str(video_path), track_result, str(sil_root))\n    log(f"[video:{video_name}] segmentation done, candidate_entries={len(sil_inputs)}")\n\n    entries = []\n    for inputs in sil_inputs:\n        frame_count = get_num_frames(inputs)\n        track_id = get_track_id(inputs)\n        sequence_name = get_sequence_name(inputs)\n        log(f"[video:{video_name}] entry track_id={track_id} sequence={sequence_name} frames={frame_count}")\n        if frame_count < min_frames:\n            log(f"[video:{video_name}] skipping track_id={track_id} because frames<{min_frames}")\n            entries.append(\n                {\n                    "entry_key": f"{video_name}:{track_id}",\n                    "video": video_name,\n                    "track_id": track_id,\n                    "sequence_name": sequence_name,\n                    "frame_count": frame_count,\n                    "status": "skipped_too_short",\n                }\n            )\n            continue\n\n        log(f"[video:{video_name}] embedding track_id={track_id}")\n        emb = extract_embedding(model, inputs)\n        log(f"[video:{video_name}] embedding done track_id={track_id} shape={list(emb.shape)}")\n        entries.append(\n            {\n                "entry_key": f"{video_name}:{track_id}",\n                "video": video_name,\n                "track_id": track_id,\n                "sequence_name": sequence_name,\n                "frame_count": frame_count,\n                "status": "ok",\n                "embedding": emb,\n            }\n        )\n    log(f"[video:{video_name}] finished, kept_entries={sum(1 for entry in entries if entry[\'status\'] == \'ok\')}")\n    return entries\n\n\ndef assign_identities(entries: List[Dict], threshold: float) -> Dict:\n    registry: List[Dict] = []\n    resolved_entries: List[Dict] = []\n\n    for entry in entries:\n        base = {\n            "entry_key": entry["entry_key"],\n            "video": entry["video"],\n            "track_id": entry["track_id"],\n            "sequence_name": entry["sequence_name"],\n            "frame_count": entry["frame_count"],\n            "status": entry["status"],\n        }\n        if entry["status"] != "ok":\n            resolved_entries.append(base)\n            continue\n\n        embedding = entry["embedding"]\n        best_match: Optional[Dict] = None\n        best_score = -1.0\n        for person in registry:\n            score = cosine_similarity(embedding, person["prototype"])\n            if score > best_score:\n                best_score = score\n                best_match = person\n\n        if best_match is None or best_score < threshold:\n            person_id = f"person_{len(registry) + 1:03d}"\n            registry.append(\n                {\n                    "person_id": person_id,\n                    "prototype": embedding.copy(),\n                    "entry_keys": [entry["entry_key"]],\n                }\n            )\n            assigned_person_id = person_id\n            assigned_score = None\n            matched_existing = False\n            log(f"[match] new identity {assigned_person_id} for {entry[\'entry_key\']}")\n        else:\n            assigned_person_id = best_match["person_id"]\n            best_match["entry_keys"].append(entry["entry_key"])\n            matched_existing = True\n            assigned_score = best_score\n            count = len(best_match["entry_keys"])\n            best_match["prototype"] = (\n                best_match["prototype"] * (count - 1) + embedding\n            ) / count\n            log(\n                f"[match] matched {entry[\'entry_key\']} -> {assigned_person_id} "\n                f"(cos={assigned_score:.4f}, threshold={threshold:.4f})"\n            )\n\n        base.update(\n            {\n                "assigned_person_id": assigned_person_id,\n                "matched_existing_person": matched_existing,\n                "best_cosine_similarity": assigned_score,\n            }\n        )\n        resolved_entries.append(base)\n\n    people = []\n    for person in registry:\n        people.append(\n            {\n                "person_id": person["person_id"],\n                "entry_count": len(person["entry_keys"]),\n                "entry_keys": person["entry_keys"],\n            }\n        )\n\n    return {\n        "summary": {\n            "processed_entries": sum(1 for entry in resolved_entries if entry["status"] == "ok"),\n            "skipped_entries": sum(1 for entry in resolved_entries if entry["status"] != "ok"),\n            "unique_people_estimated": len(people),\n            "total_entries_estimated": sum(1 for entry in resolved_entries if entry["status"] == "ok"),\n            "cosine_threshold": threshold,\n        },\n        "people": people,\n        "entries": resolved_entries,\n    }\n\n\ndef main():\n    parser = argparse.ArgumentParser(\n        description="Probe-only gait entry analysis for a folder of videos."\n    )\n    parser.add_argument(\n        "--video-dir",\n        default=str(PROJECT_ROOT / "team2videos"),\n        help="Folder containing input videos.",\n    )\n    parser.add_argument(\n        "--video-path",\n        default=None,\n        help="Optional path to a single input video. Overrides --video-dir.",\n    )\n    parser.add_argument(\n        "--output-json",\n        default=str(REPO_ROOT / "output" / "probe_only_entry_analysis.json"),\n        help="Where to save the summary JSON.",\n    )\n    parser.add_argument(\n        "--work-root",\n        default=str(REPO_ROOT / "demo" / "output" / "probe_only"),\n        help="Working folder for tracking videos and silhouettes.",\n    )\n    parser.add_argument(\n        "--cosine-threshold",\n        type=float,\n        default=0.75,\n        help="Cosine similarity threshold for matching a new entry to an existing person.",\n    )\n    parser.add_argument(\n        "--min-frames",\n        type=int,\n        default=20,\n        help="Minimum number of silhouette frames required to keep an entry.",\n    )\n    parser.add_argument(\n        "--model",\n        choices=["grew_gaitbase", "current_gaitbase", "grew_gaitgl"],\n        default="grew_gaitbase",\n        help="Which gait checkpoint/profile to use.",\n    )\n    args = parser.parse_args()\n\n    output_json = Path(args.output_json).resolve()\n    work_root = Path(args.work_root).resolve()\n    if args.video_path is not None:\n        single_video = Path(args.video_path).resolve()\n        if not single_video.exists():\n            raise FileNotFoundError(f"Video file does not exist: {single_video}")\n        video_dir = single_video.parent\n        video_paths = [single_video]\n    else:\n        video_dir = Path(args.video_dir).resolve()\n        if not video_dir.exists():\n            raise FileNotFoundError(f"Video folder does not exist: {video_dir}")\n        video_paths = sorted(\n            [path for path in video_dir.iterdir() if path.suffix.lower() in {".mp4", ".avi", ".mov", ".mkv"}]\n        )\n        if not video_paths:\n            raise FileNotFoundError(f"No videos found in {video_dir}")\n\n    model, profile = build_model(args.model)\n    log(f"[run] total_videos={len(video_paths)} model={args.model}")\n\n    all_entries: List[Dict] = []\n    for video_path in video_paths:\n        log(f"[run] analyzing {video_path.name}")\n        all_entries.extend(\n            analyze_video(\n                model=model,\n                video_path=video_path,\n                work_root=work_root,\n                min_frames=args.min_frames,\n            )\n        )\n\n    log("[run] assigning identities")\n    results = assign_identities(all_entries, args.cosine_threshold)\n    results["config"] = {\n        "video_dir": str(video_dir),\n        "work_root": str(work_root),\n        "output_json": str(output_json),\n        "model": profile["display_name"],\n        "model_key": args.model,\n        "checkpoint": str(profile["checkpoint"]),\n        "config_path": str(profile["config"]),\n        "notes": [\n            "Each stable track is treated as one entry candidate.",\n            "This is a first-pass probe-only matcher without a fixed gallery.",\n            "The gait model still operates on silhouettes generated from RGB video.",\n        ],\n    }\n\n    output_json.parent.mkdir(parents=True, exist_ok=True)\n    output_json.write_text(json.dumps(results, indent=2))\n    log(json.dumps(results["summary"], indent=2))\n    log(f"[run] saved analysis JSON to {output_json}")\n\n\nif __name__ == "__main__":\n    main()\n')
print('wrote', path)
path = ROOT / 'clean_demo_v2/tools/build_two_identity_gallery.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('import argparse\nimport json\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\n\nSCRIPT_DIR = Path(__file__).resolve().parent\nPROJECT_ROOT = SCRIPT_DIR.parents[1]\nsys.path.insert(0, str(PROJECT_ROOT / "OpenGait" / "tools"))\n\nfrom probe_only_entry_analysis import analyze_video, build_model, log  # noqa: E402\n\n\ndef process_gallery_video(model, video_path: Path, label: str, work_root: Path, min_frames: int):\n    entries = analyze_video(\n        model=model,\n        video_path=video_path,\n        work_root=work_root / label,\n        min_frames=min_frames,\n    )\n    ok_entries = [entry for entry in entries if entry["status"] == "ok"]\n    if not ok_entries:\n        raise ValueError(f"No valid gallery entries were extracted for {label}: {video_path}")\n    embeddings = np.stack([entry["embedding"].astype(np.float32) for entry in ok_entries], axis=0)\n    prototype = embeddings.mean(axis=0)\n    metadata = {\n        "label": label,\n        "video": str(video_path),\n        "valid_entries": len(ok_entries),\n        "entries": [\n            {\n                "entry_key": entry["entry_key"],\n                "track_id": entry["track_id"],\n                "frame_count": entry["frame_count"],\n                "embedding_norm": float(np.linalg.norm(entry["embedding"])),\n            }\n            for entry in ok_entries\n        ],\n    }\n    return prototype, metadata\n\n\ndef main():\n    parser = argparse.ArgumentParser(\n        description="Build a two-person Pritom/Coco gallery from clean_demo_v2 gallery videos."\n    )\n    parser.add_argument("--pritom-video", required=True)\n    parser.add_argument("--coco-video", required=True)\n    parser.add_argument("--gallery-out", required=True)\n    parser.add_argument("--metadata-out", required=True)\n    parser.add_argument("--work-root", required=True)\n    parser.add_argument("--model", choices=["grew_gaitbase", "current_gaitbase", "grew_gaitgl"], default="grew_gaitbase")\n    parser.add_argument("--min-frames", type=int, default=20)\n    args = parser.parse_args()\n\n    pritom_video = Path(args.pritom_video).resolve()\n    coco_video = Path(args.coco_video).resolve()\n    gallery_out = Path(args.gallery_out).resolve()\n    metadata_out = Path(args.metadata_out).resolve()\n    work_root = Path(args.work_root).resolve()\n\n    model, profile = build_model(args.model)\n    pritom_embedding, pritom_meta = process_gallery_video(model, pritom_video, "pritom", work_root, args.min_frames)\n    coco_embedding, coco_meta = process_gallery_video(model, coco_video, "coco", work_root, args.min_frames)\n\n    embeddings = np.stack([pritom_embedding, coco_embedding], axis=0).astype(np.float32)\n    labels = np.array(["pritom", "coco"])\n    entry_keys = np.array(["pritomgallery:gallery", "cocogallery:gallery"])\n    video_names = np.array([pritom_video.stem, coco_video.stem])\n\n    gallery_out.parent.mkdir(parents=True, exist_ok=True)\n    np.savez_compressed(\n        gallery_out,\n        embeddings=embeddings,\n        labels=labels,\n        entry_keys=entry_keys,\n        video_names=video_names,\n    )\n\n    pritom_flat = pritom_embedding.reshape(-1)\n    coco_flat = coco_embedding.reshape(-1)\n    cosine = float(np.dot(pritom_flat, coco_flat) / (np.linalg.norm(pritom_flat) * np.linalg.norm(coco_flat)))\n    metadata = {\n        "model": profile["display_name"],\n        "model_key": args.model,\n        "gallery_out": str(gallery_out),\n        "identities": {\n            "pritom": pritom_meta,\n            "coco": coco_meta,\n        },\n        "pritom_vs_coco_cosine": cosine,\n    }\n    metadata_out.parent.mkdir(parents=True, exist_ok=True)\n    metadata_out.write_text(json.dumps(metadata, indent=2))\n    log(json.dumps({"gallery_out": str(gallery_out), "pritom_vs_coco_cosine": cosine}, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n')
print('wrote', path)
path = ROOT / 'clean_demo_v2/tools/identify_probe_with_gallery.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('import argparse\nimport json\nfrom collections import Counter\nfrom pathlib import Path\nimport sys\nfrom typing import Dict, List\n\nimport numpy as np\n\nSCRIPT_DIR = Path(__file__).resolve().parent\nPROJECT_ROOT = SCRIPT_DIR.parents[1]\nsys.path.insert(0, str(PROJECT_ROOT / "OpenGait" / "tools"))\n\nfrom probe_only_entry_analysis import (\n    PROJECT_ROOT,\n    REPO_ROOT,\n    analyze_video,\n    build_model,\n    cosine_similarity,\n    log,\n)\n\n\ndef load_gallery(path: Path) -> Dict:\n    data = np.load(path)\n    return {\n        "embeddings": data["embeddings"].astype(np.float32),\n        "labels": data["labels"].astype(str),\n        "entry_keys": data["entry_keys"].astype(str),\n        "video_names": data["video_names"].astype(str),\n    }\n\n\ndef score_identity(probe_embedding: np.ndarray, gallery_embeddings: np.ndarray, mode: str, top_k: int) -> float:\n    scores = np.array([cosine_similarity(probe_embedding, emb) for emb in gallery_embeddings], dtype=np.float32)\n    if scores.size == 0:\n        return -1.0\n    if mode == "max":\n        return float(np.max(scores))\n    if mode == "mean":\n        return float(np.mean(scores))\n    if mode == "topk":\n        k = min(top_k, scores.size)\n        return float(np.mean(np.sort(scores)[-k:]))\n    raise ValueError(f"Unsupported match mode: {mode}")\n\n\ndef identify_entry(\n    embedding: np.ndarray,\n    gallery: Dict,\n    match_mode: str,\n    top_k: int,\n    threshold: float,\n    margin: float,\n) -> Dict:\n    identity_scores = {}\n    for identity in sorted(set(gallery["labels"])):\n        mask = gallery["labels"] == identity\n        identity_scores[identity] = score_identity(\n            embedding,\n            gallery["embeddings"][mask],\n            mode=match_mode,\n            top_k=top_k,\n        )\n\n    ranked = sorted(identity_scores.items(), key=lambda item: item[1], reverse=True)\n    best_identity, best_score = ranked[0]\n    second_identity, second_score = ranked[1] if len(ranked) > 1 else ("none", -1.0)\n    score_margin = best_score - second_score\n    accepted = best_score >= threshold and score_margin >= margin\n\n    return {\n        "assigned_identity": best_identity if accepted else "unknown",\n        "accepted": accepted,\n        "best_identity": best_identity,\n        "best_score": float(best_score),\n        "second_identity": second_identity,\n        "second_score": float(second_score),\n        "score_margin": float(score_margin),\n        "identity_scores": {key: float(value) for key, value in identity_scores.items()},\n    }\n\n\ndef main():\n    parser = argparse.ArgumentParser(\n        description="Identify entries in probe videos using a multi-identity gait gallery."\n    )\n    parser.add_argument("--gallery-npz", default=str(REPO_ROOT / "output" / "multi_identity_gallery.npz"))\n    parser.add_argument("--video-dir", default=str(PROJECT_ROOT / "newvideos"))\n    parser.add_argument("--video-path", default=None, help="Optional single probe video. Defaults to Test1/Test2.")\n    parser.add_argument("--output-json", default=str(REPO_ROOT / "output" / "multi_gallery_probe_results.json"))\n    parser.add_argument("--work-root", default=str(REPO_ROOT / "demo" / "output" / "multi_gallery_probe"))\n    parser.add_argument("--model", choices=["grew_gaitbase", "current_gaitbase", "grew_gaitgl"], default="grew_gaitbase")\n    parser.add_argument("--min-frames", type=int, default=20)\n    parser.add_argument("--threshold", type=float, default=0.97)\n    parser.add_argument("--margin", type=float, default=0.005)\n    parser.add_argument("--match-mode", choices=["max", "mean", "topk"], default="max")\n    parser.add_argument("--top-k", type=int, default=3)\n    args = parser.parse_args()\n\n    gallery_path = Path(args.gallery_npz).resolve()\n    video_dir = Path(args.video_dir).resolve()\n    output_json = Path(args.output_json).resolve()\n    work_root = Path(args.work_root).resolve()\n\n    if not gallery_path.exists():\n        raise FileNotFoundError(f"Gallery file does not exist: {gallery_path}")\n\n    if args.video_path:\n        video_paths = [Path(args.video_path).resolve()]\n    else:\n        video_paths = [video_dir / "Test1.mp4", video_dir / "Test2.mp4"]\n        missing = [path for path in video_paths if not path.exists()]\n        if missing:\n            raise FileNotFoundError("Missing probe videos: " + ", ".join(str(path) for path in missing))\n\n    gallery = load_gallery(gallery_path)\n    model, profile = build_model(args.model)\n\n    all_entries: List[Dict] = []\n    for video_path in video_paths:\n        log(f"[probe] analyzing {video_path.name}")\n        all_entries.extend(\n            analyze_video(\n                model=model,\n                video_path=video_path,\n                work_root=work_root,\n                min_frames=args.min_frames,\n            )\n        )\n\n    resolved_entries = []\n    for entry in all_entries:\n        item = {\n            "entry_key": entry["entry_key"],\n            "video": entry["video"],\n            "track_id": entry["track_id"],\n            "sequence_name": entry["sequence_name"],\n            "frame_count": entry["frame_count"],\n            "status": entry["status"],\n        }\n        if entry["status"] == "ok":\n            item.update(\n                identify_entry(\n                    entry["embedding"],\n                    gallery=gallery,\n                    match_mode=args.match_mode,\n                    top_k=args.top_k,\n                    threshold=args.threshold,\n                    margin=args.margin,\n                )\n            )\n        resolved_entries.append(item)\n\n    counts = Counter(\n        entry["assigned_identity"]\n        for entry in resolved_entries\n        if entry["status"] == "ok"\n    )\n    per_video_counts = {}\n    for entry in resolved_entries:\n        if entry["status"] != "ok":\n            continue\n        per_video_counts.setdefault(entry["video"], Counter())\n        per_video_counts[entry["video"]][entry["assigned_identity"]] += 1\n\n    results = {\n        "config": {\n            "gallery_npz": str(gallery_path),\n            "video_paths": [str(path) for path in video_paths],\n            "work_root": str(work_root),\n            "output_json": str(output_json),\n            "model": profile["display_name"],\n            "model_key": args.model,\n            "threshold": args.threshold,\n            "margin": args.margin,\n            "match_mode": args.match_mode,\n            "top_k": args.top_k,\n            "gallery_identities": sorted(set(gallery["labels"])),\n        },\n        "summary": {\n            "processed_entries": sum(1 for entry in resolved_entries if entry["status"] == "ok"),\n            "skipped_entries": sum(1 for entry in resolved_entries if entry["status"] != "ok"),\n            "counts": dict(counts),\n            "per_video_counts": {video: dict(counter) for video, counter in per_video_counts.items()},\n        },\n        "entries": resolved_entries,\n    }\n\n    output_json.parent.mkdir(parents=True, exist_ok=True)\n    output_json.write_text(json.dumps(results, indent=2))\n    log(json.dumps(results["summary"], indent=2))\n    log(f"[probe] saved multi-gallery probe results to {output_json}")\n\n\nif __name__ == "__main__":\n    main()\n')
print('wrote', path)
path = ROOT / 'clean_demo_v2/tools/render_closed_set_annotations.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('import argparse\nimport json\nfrom collections import Counter\nfrom pathlib import Path\nfrom typing import Dict, List\n\nimport cv2\n\n\ndef load_tracking_file(path: Path) -> Dict[int, List[Dict]]:\n    frame_tracks: Dict[int, List[Dict]] = {}\n    for line in path.read_text().splitlines():\n        parts = line.split(",")\n        if len(parts) < 6:\n            continue\n        frame_id = int(float(parts[0]))\n        track_id = int(float(parts[1]))\n        frame_tracks.setdefault(frame_id, []).append(\n            {\n                "track_id": f"{track_id:03d}",\n                "bbox": [float(parts[2]), float(parts[3]), float(parts[4]), float(parts[5])],\n            }\n        )\n    return frame_tracks\n\n\ndef color_for_label(label: str):\n    colors = {\n        "pritom": (0, 220, 0),\n        "coco": (255, 140, 0),\n        "jeevith": (0, 165, 255),\n        "unknown": (180, 180, 180),\n    }\n    return colors.get(label, (255, 255, 255))\n\n\ndef draw_label(frame, x: int, y: int, text: str, color):\n    font = cv2.FONT_HERSHEY_SIMPLEX\n    scale = 0.72\n    thickness = 2\n    (tw, th), baseline = cv2.getTextSize(text, font, scale, thickness)\n    y0 = max(0, y - th - baseline - 8)\n    cv2.rectangle(frame, (x, y0), (x + tw + 8, y0 + th + baseline + 8), color, -1)\n    cv2.putText(frame, text, (x + 4, y0 + th + 2), font, scale, (0, 0, 0), thickness, cv2.LINE_AA)\n\n\ndef build_decisions(result_json: Path) -> Dict[str, Dict]:\n    payload = json.loads(result_json.read_text())\n    counts = Counter()\n    decisions = {}\n    for entry in sorted(payload["entries"], key=lambda item: item["track_id"]):\n        if entry["status"] != "ok":\n            continue\n        label = entry["assigned_identity"]\n        counts[label] += 1\n        decisions[entry["track_id"]] = {\n            "label": label,\n            "count": counts[label],\n            "score": entry["best_score"],\n            "second_identity": entry["second_identity"],\n            "second_score": entry["second_score"],\n        }\n    return decisions\n\n\ndef main():\n    parser = argparse.ArgumentParser(\n        description="Render a closed-set annotated video from an existing result JSON and tracking TXT."\n    )\n    parser.add_argument("--video-path", required=True)\n    parser.add_argument("--tracking-txt", required=True)\n    parser.add_argument("--result-json", required=True)\n    parser.add_argument("--output-video", required=True)\n    parser.add_argument("--title", default="Closed-set gait recognition")\n    args = parser.parse_args()\n\n    video_path = Path(args.video_path).resolve()\n    tracking_txt = Path(args.tracking_txt).resolve()\n    result_json = Path(args.result_json).resolve()\n    output_video = Path(args.output_video).resolve()\n\n    frame_tracks = load_tracking_file(tracking_txt)\n    decisions = build_decisions(result_json)\n\n    cap = cv2.VideoCapture(str(video_path))\n    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0\n    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))\n    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))\n    output_video.parent.mkdir(parents=True, exist_ok=True)\n    writer = cv2.VideoWriter(str(output_video), cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))\n\n    frame_id = 0\n    while True:\n        ok, frame = cap.read()\n        if not ok:\n            break\n        cv2.putText(frame, args.title, (25, 45), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 3, cv2.LINE_AA)\n        for item in frame_tracks.get(frame_id, []):\n            decision = decisions.get(item["track_id"])\n            if decision is None:\n                continue\n            x, y, w, h = item["bbox"]\n            x1, y1 = int(x), int(y)\n            x2, y2 = int(x + w), int(y + h)\n            color = color_for_label(decision["label"])\n            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)\n            text = f"{decision[\'label\']} count={decision[\'count\']} score={decision[\'score\']:.3f}"\n            draw_label(frame, x1, max(0, y1 - 4), text, color)\n        writer.write(frame)\n        frame_id += 1\n\n    cap.release()\n    writer.release()\n    print(f"[render] saved annotated video to {output_video}", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n')
print('wrote', path)
path = ROOT / 'clean_demo_v2/gradio_app.py'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('import json\nimport shutil\nimport subprocess\nimport sys\nimport time\nfrom collections import Counter\nfrom pathlib import Path\nfrom typing import Dict, List\n\nimport cv2\nimport gradio as gr\nimport numpy as np\n\n\nROOT = Path(__file__).resolve().parents[1]\nCLEAN_ROOT = ROOT / "clean_demo_v2"\nOPENGAIT_TOOLS = ROOT / "OpenGait" / "tools"\nsys.path.insert(0, str(OPENGAIT_TOOLS))\n\nfrom probe_only_entry_analysis import analyze_video, build_model, cosine_similarity, log  # noqa: E402\n\n\ndef copy_upload(src, dst: Path):\n    src_path = Path(src)\n    dst.parent.mkdir(parents=True, exist_ok=True)\n    shutil.copy(src_path, dst)\n    return dst\n\n\ndef timestamped_run_root() -> Path:\n    ts = time.strftime("%Y%m%d_%H%M%S")\n    run_root = CLEAN_ROOT / "output" / f"gradio_multi_gallery_{ts}"\n    run_root.mkdir(parents=True, exist_ok=True)\n    return run_root\n\n\ndef load_tracking_file(path: Path) -> Dict[int, List[Dict]]:\n    frame_tracks: Dict[int, List[Dict]] = {}\n    if not path.exists():\n        raise FileNotFoundError(f"Tracking file not found: {path}")\n    for line in path.read_text().splitlines():\n        parts = line.split(",")\n        if len(parts) < 6:\n            continue\n        frame_id = int(float(parts[0]))\n        track_id = int(float(parts[1]))\n        frame_tracks.setdefault(frame_id, []).append(\n            {\n                "track_id": f"{track_id:03d}",\n                "bbox": [float(parts[2]), float(parts[3]), float(parts[4]), float(parts[5])],\n            }\n        )\n    return frame_tracks\n\n\ndef color_for_label(label: str):\n    if label == "unknown":\n        return (150, 150, 150)\n    try:\n        idx = int(label.replace("person", ""))\n    except ValueError:\n        idx = 1\n    return ((37 * idx) % 255, (137 * idx) % 255, (211 * idx) % 255)\n\n\ndef draw_label(frame, x: int, y: int, text: str, color):\n    font = cv2.FONT_HERSHEY_SIMPLEX\n    scale = 0.72\n    thickness = 2\n    (tw, th), baseline = cv2.getTextSize(text, font, scale, thickness)\n    y0 = max(0, y - th - baseline - 8)\n    cv2.rectangle(frame, (x, y0), (x + tw + 8, y0 + th + baseline + 8), color, -1)\n    cv2.putText(frame, text, (x + 4, y0 + th + 2), font, scale, (0, 0, 0), thickness, cv2.LINE_AA)\n\n\ndef clip_video(input_path: Path, output_path: Path, max_seconds: float) -> Path:\n    if max_seconds <= 0:\n        return input_path\n\n    cap = cv2.VideoCapture(str(input_path))\n    if not cap.isOpened():\n        raise RuntimeError(f"Could not open probe video for clipping: {input_path}")\n\n    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0\n    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))\n    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))\n    max_frames = int(max_seconds * fps)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    writer = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))\n\n    written = 0\n    while written < max_frames:\n        ok, frame = cap.read()\n        if not ok:\n            break\n        writer.write(frame)\n        written += 1\n\n    cap.release()\n    writer.release()\n    if written == 0:\n        raise RuntimeError("Probe clipping produced zero frames.")\n    return output_path\n\n\ndef render_annotations(\n    video_path: Path,\n    tracking_txt: Path,\n    decisions: List[Dict],\n    output_video: Path,\n    title: str,\n    max_side: int = 720,\n    target_fps: float = 24.0,\n):\n    decision_by_track = {item["track_id"]: item for item in decisions}\n    frame_tracks = load_tracking_file(tracking_txt)\n\n    cap = cv2.VideoCapture(str(video_path))\n    input_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0\n    stride = max(1, int(round(input_fps / target_fps)))\n    fps = input_fps / stride\n    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))\n    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))\n    largest_side = max(width, height)\n    if largest_side > max_side:\n        scale = max_side / largest_side\n        out_width = int(width * scale)\n        out_height = int(height * scale)\n    else:\n        out_width = width\n        out_height = height\n    output_video.parent.mkdir(parents=True, exist_ok=True)\n    writer = cv2.VideoWriter(str(output_video), cv2.VideoWriter_fourcc(*"mp4v"), fps, (out_width, out_height))\n\n    frame_id = 0\n    written_frames = 0\n    while True:\n        ok, frame = cap.read()\n        if not ok:\n            break\n        if frame_id % stride != 0:\n            frame_id += 1\n            continue\n        cv2.putText(frame, title, (25, 45), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 3, cv2.LINE_AA)\n        for item in frame_tracks.get(frame_id, []):\n            decision = decision_by_track.get(item["track_id"])\n            if decision is None:\n                continue\n            x, y, w, h = item["bbox"]\n            x1, y1 = int(x), int(y)\n            x2, y2 = int(x + w), int(y + h)\n            label = decision["assigned_identity"]\n            color = color_for_label(label)\n            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)\n            text = f"{label} count={decision[\'display_count\']} score={decision[\'best_score\']:.3f}"\n            draw_label(frame, x1, max(0, y1 - 4), text, color)\n        if (out_width, out_height) != (width, height):\n            frame = cv2.resize(frame, (out_width, out_height), interpolation=cv2.INTER_AREA)\n        writer.write(frame)\n        written_frames += 1\n        frame_id += 1\n\n    cap.release()\n    writer.release()\n    if written_frames == 0:\n        raise RuntimeError("Annotated video rendering produced zero frames.")\n\n\ndef make_browser_playable_video(input_video: Path, output_video: Path) -> Path:\n    """Convert OpenCV MP4 output to browser-friendly H.264 if ffmpeg is available."""\n    try:\n        import imageio_ffmpeg\n\n        ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()\n    except Exception:\n        ffmpeg = shutil.which("ffmpeg")\n\n    if not ffmpeg:\n        return input_video\n\n    output_video.parent.mkdir(parents=True, exist_ok=True)\n    cmd = [\n        ffmpeg,\n        "-y",\n        "-i",\n        str(input_video),\n        "-vcodec",\n        "libx264",\n        "-pix_fmt",\n        "yuv420p",\n        "-movflags",\n        "+faststart",\n        "-an",\n        str(output_video),\n    ]\n    proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)\n    if proc.returncode != 0 or not output_video.exists() or output_video.stat().st_size == 0:\n        return input_video\n    return output_video\n\n\ndef build_gallery(model, gallery_files: List[str], run_root: Path, min_frames: int, progress=None):\n    gallery = []\n    metadata = []\n    total = max(len(gallery_files), 1)\n    for idx, gallery_file in enumerate(gallery_files, start=1):\n        label = f"person{idx}"\n        if progress is not None:\n            progress(\n                0.10 + 0.45 * ((idx - 1) / total),\n                desc=f"Building gallery embedding for {label}",\n            )\n        copied_path = copy_upload(\n            gallery_file,\n            run_root / "inputs" / "gallery" / f"{label}{Path(gallery_file).suffix or \'.mp4\'}",\n        )\n        log(f"[gradio] building gallery for {label}: {copied_path}")\n        entries = analyze_video(\n            model=model,\n            video_path=copied_path,\n            work_root=run_root / "work" / "gallery" / label,\n            min_frames=min_frames,\n        )\n        ok_entries = [entry for entry in entries if entry["status"] == "ok"]\n        if not ok_entries:\n            raise gr.Error(f"No valid walking sequence found in gallery video for {label}.")\n        embeddings = np.stack([entry["embedding"].astype(np.float32) for entry in ok_entries], axis=0)\n        prototype = embeddings.mean(axis=0)\n        gallery.append(\n            {\n                "label": label,\n                "source_file": str(copied_path),\n                "embedding": prototype,\n                "valid_entries": len(ok_entries),\n            }\n        )\n        metadata.append(\n            {\n                "label": label,\n                "source_file": str(copied_path),\n                "valid_entries": len(ok_entries),\n                "entry_keys": [entry["entry_key"] for entry in ok_entries],\n                "frame_counts": [entry["frame_count"] for entry in ok_entries],\n            }\n        )\n        if progress is not None:\n            progress(\n                0.10 + 0.45 * (idx / total),\n                desc=f"Finished gallery embedding for {label}",\n            )\n    return gallery, metadata\n\n\ndef score_probe_to_gallery(embedding: np.ndarray, gallery: List[Dict]) -> List[Dict]:\n    scores = []\n    for item in gallery:\n        scores.append(\n            {\n                "label": item["label"],\n                "score": cosine_similarity(embedding, item["embedding"]),\n            }\n        )\n    return sorted(scores, key=lambda item: item["score"], reverse=True)\n\n\ndef assign_probe_entries(\n    probe_entries: List[Dict],\n    gallery: List[Dict],\n    threshold: float,\n) -> List[Dict]:\n    decisions = []\n    counts = Counter()\n    closed_set = len(gallery) >= 2\n\n    for entry in sorted(probe_entries, key=lambda item: item["track_id"]):\n        if entry["status"] != "ok":\n            decisions.append(\n                {\n                    "entry_key": entry["entry_key"],\n                    "video": entry["video"],\n                    "track_id": entry["track_id"],\n                    "frame_count": entry["frame_count"],\n                    "status": entry["status"],\n                    "assigned_identity": "skipped",\n                }\n            )\n            continue\n\n        scores = score_probe_to_gallery(entry["embedding"], gallery)\n        best = scores[0]\n        second = scores[1] if len(scores) > 1 else {"label": "none", "score": -1.0}\n\n        if closed_set:\n            assigned = best["label"]\n        else:\n            assigned = best["label"] if best["score"] >= threshold else "unknown"\n\n        counts[assigned] += 1\n        decisions.append(\n            {\n                "entry_key": entry["entry_key"],\n                "video": entry["video"],\n                "track_id": entry["track_id"],\n                "frame_count": entry["frame_count"],\n                "status": "ok",\n                "assigned_identity": assigned,\n                "display_count": counts[assigned],\n                "best_score": float(best["score"]),\n                "best_identity": best["label"],\n                "second_identity": second["label"],\n                "second_score": float(second["score"]),\n                "all_scores": {item["label"]: float(item["score"]) for item in scores},\n            }\n        )\n    return decisions\n\n\ndef format_ui_summary(ui_summary: Dict) -> str:\n    lines = []\n    mode_label = (\n        "Closed-set gallery matching"\n        if ui_summary["mode"] == "closed_set_best_match"\n        else "Single-gallery threshold fallback"\n    )\n    lines.append("Gait Recognition Result")\n    lines.append("=" * 24)\n    lines.append(f"Mode: {mode_label}")\n    lines.append(f"Model: {ui_summary[\'model\']}")\n    lines.append("")\n    lines.append("Gallery Mapping")\n    for label, filename in ui_summary["gallery_mapping"].items():\n        lines.append(f"- {label}: {filename}")\n    lines.append("")\n    lines.append("Entry Counts")\n    for label, count in sorted(ui_summary["counts"].items()):\n        lines.append(f"- {label}: {count}")\n    lines.append("")\n    lines.append("Processing Summary")\n    lines.append(f"- Processed entries: {ui_summary[\'processed_entries\']}")\n    lines.append(f"- Skipped entries: {ui_summary[\'skipped_entries\']}")\n    lines.append("")\n    lines.append("Output")\n    lines.append(f"- Run folder: {ui_summary[\'run_root\']}")\n    return "\\n".join(lines)\n\n\ndef run_multi_gallery(gallery_files, probe_video, model_name, threshold, min_frames, max_probe_seconds, progress=gr.Progress()):\n    if not gallery_files:\n        raise gr.Error("Please upload at least one gallery video.")\n    if probe_video is None:\n        raise gr.Error("Please upload one probe video.")\n\n    run_root = timestamped_run_root()\n    progress(0.02, desc="Loading selected gait model")\n    model, profile = build_model(model_name)\n\n    progress(0.08, desc="Preparing gallery videos")\n    gallery, gallery_metadata = build_gallery(\n        model=model,\n        gallery_files=gallery_files,\n        run_root=run_root,\n        min_frames=min_frames,\n        progress=progress,\n    )\n\n    progress(0.58, desc="Copying probe video")\n    probe_path = copy_upload(\n        probe_video,\n        run_root / "inputs" / "probe" / f"probe{Path(probe_video).suffix or \'.mp4\'}",\n    )\n    if max_probe_seconds and max_probe_seconds > 0:\n        progress(0.60, desc=f"Clipping probe to first {max_probe_seconds:.1f} seconds")\n        probe_path = clip_video(\n            probe_path,\n            run_root / "inputs" / "probe" / f"probe_first_{int(max_probe_seconds)}s.mp4",\n            float(max_probe_seconds),\n        )\n\n    log(f"[gradio] analyzing probe: {probe_path}")\n    progress(0.62, desc="Tracking, segmenting, and extracting probe embeddings")\n    probe_entries = analyze_video(\n        model=model,\n        video_path=probe_path,\n        work_root=run_root / "work" / "probe",\n        min_frames=min_frames,\n    )\n    progress(0.82, desc="Matching probe entries against gallery")\n    decisions = assign_probe_entries(probe_entries, gallery, threshold=threshold)\n    counts = Counter(\n        decision["assigned_identity"]\n        for decision in decisions\n        if decision["status"] == "ok"\n    )\n\n    tracking_txt = run_root / "work" / "probe" / "tracking" / probe_path.stem / f"{probe_path.stem}.txt"\n    annotated_video_raw = run_root / "annotated_probe_display_raw.mp4"\n    annotated_video = run_root / "annotated_probe_display.mp4"\n    mode = "closed_set_best_match" if len(gallery) >= 2 else "target_vs_unknown_threshold"\n    title = "Gait recognition: closed set" if len(gallery) >= 2 else f"Gait recognition: threshold={threshold:.3f}"\n    progress(0.90, desc="Rendering annotated output video")\n    render_annotations(probe_path, tracking_txt, decisions, annotated_video_raw, title)\n    progress(0.96, desc="Converting annotated video for browser playback")\n    annotated_video = make_browser_playable_video(annotated_video_raw, annotated_video)\n\n    result = {\n        "config": {\n            "mode": mode,\n            "model": profile["display_name"],\n            "model_key": model_name,\n            "threshold_if_single_gallery": threshold,\n            "min_frames": min_frames,\n            "max_probe_seconds": max_probe_seconds,\n            "run_root": str(run_root),\n            "probe_video": str(probe_path),\n        },\n        "gallery": gallery_metadata,\n        "summary": {\n            "gallery_identity_count": len(gallery),\n            "processed_entries": sum(1 for decision in decisions if decision["status"] == "ok"),\n            "skipped_entries": sum(1 for decision in decisions if decision["status"] != "ok"),\n            "counts": dict(counts),\n        },\n        "entries": decisions,\n    }\n\n    result_json = run_root / "result.json"\n    summary_txt = run_root / "summary.txt"\n    result_json.write_text(json.dumps(result, indent=2))\n    summary_txt.write_text(\n        "\\n".join(\n            [\n                "Generic multi-gallery gait recognition summary",\n                f"Mode: {mode}",\n                f"Model: {profile[\'display_name\']}",\n                f"Counts: {dict(counts)}",\n                f"Processed entries: {result[\'summary\'][\'processed_entries\']}",\n                f"Skipped entries: {result[\'summary\'][\'skipped_entries\']}",\n                f"Run folder: {run_root}",\n                f"Annotated video: {annotated_video}",\n            ]\n        )\n    )\n\n    ui_summary = {\n        "mode": mode,\n        "model": model_name,\n        "counts": dict(counts),\n        "processed_entries": result["summary"]["processed_entries"],\n        "skipped_entries": result["summary"]["skipped_entries"],\n        "gallery_mapping": {\n            item["label"]: Path(item["source_file"]).name\n            for item in gallery_metadata\n        },\n        "run_root": str(run_root),\n        "probe_duration_limit_seconds": max_probe_seconds,\n    }\n    progress(1.0, desc="Done")\n    return format_ui_summary(ui_summary), str(annotated_video), str(result_json), str(summary_txt)\n\n\nwith gr.Blocks(title="OpenGait Gallery/Probe Recognition") as demo:\n    gr.Markdown(\n        """\n        # OpenGait Gallery/Probe Recognition\n\n        Upload **any number of gallery videos** and one probe video.\n\n        The app automatically labels gallery videos by upload order:\n        `person1`, `person2`, `person3`, ...\n\n        - If you upload **2 or more gallery videos**, the app runs closed-set recognition: every probe entry is assigned to the most similar gallery person.\n        - If you upload **1 gallery video**, the app falls back to target-vs-unknown thresholding using `person1` vs `unknown`.\n\n        The output includes counts and an annotated overlay video.\n        """\n    )\n\n    with gr.Row():\n        model = gr.Dropdown(\n            choices=["grew_gaitbase", "grew_gaitgl", "current_gaitbase"],\n            value="grew_gaitbase",\n            label="Model",\n        )\n        threshold = gr.Slider(\n            0.90,\n            0.995,\n            value=0.97,\n            step=0.001,\n            label="Threshold, used only when one gallery video is uploaded",\n        )\n        min_frames = gr.Slider(\n            5,\n            80,\n            value=20,\n            step=1,\n            label="Minimum valid silhouette frames per entry",\n        )\n        max_probe_seconds = gr.Slider(\n            0,\n            180,\n            value=0,\n            step=1,\n            label="Max probe duration in seconds, 0 means full video",\n        )\n\n    gallery_files = gr.File(\n        label="Gallery videos, upload one or more",\n        file_count="multiple",\n        file_types=["video"],\n    )\n    probe_file = gr.File(label="Probe video", file_types=["video"])\n    run_button = gr.Button("Run Recognition", variant="primary")\n\n    summary = gr.Textbox(label="Counts and run summary", lines=18)\n    annotated = gr.Video(label="Annotated output video", format="mp4", height=420, width=520, interactive=False)\n    result_file = gr.File(label="Download result JSON")\n    summary_file = gr.File(label="Download summary TXT")\n\n    run_button.click(\n        run_multi_gallery,\n        inputs=[gallery_files, probe_file, model, threshold, min_frames, max_probe_seconds],\n        outputs=[summary, annotated, result_file, summary_file],\n    )\n\n\nif __name__ == "__main__":\n    demo.queue(max_size=1).launch(share=True)\n')
print('wrote', path)
path = ROOT / 'clean_demo_v2/run_pritom_coco_test1_closed.sh'
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('#!/usr/bin/env bash\nset -euo pipefail\n\nROOT="$(cd "$(dirname "${BASH_SOURCE[0]}")/.." && pwd)"\ncd "$ROOT"\n\nTS="$(date +%Y%m%d_%H%M%S)"\nCLEAN_ROOT="$ROOT/clean_demo_v2"\nOUT_ROOT="$CLEAN_ROOT/output/pritom_coco_test1_closed_$TS"\nWORK_ROOT="$OUT_ROOT/work"\n\nMODEL="${MODEL:-grew_gaitbase}"\n\nmkdir -p "$OUT_ROOT"\n\necho "[pritom-coco-closed] timestamp=$TS"\necho "[pritom-coco-closed] model=$MODEL"\necho "[pritom-coco-closed] gallery pritom: $CLEAN_ROOT/gallery/pritomgallery.mp4"\necho "[pritom-coco-closed] gallery coco:   $CLEAN_ROOT/gallery/cocogallery.mp4"\necho "[pritom-coco-closed] probe:          $CLEAN_ROOT/probes/test1probe.mp4"\n\npython clean_demo_v2/tools/build_two_identity_gallery.py \\\n  --pritom-video "$CLEAN_ROOT/gallery/pritomgallery.mp4" \\\n  --coco-video "$CLEAN_ROOT/gallery/cocogallery.mp4" \\\n  --model "$MODEL" \\\n  --work-root "$WORK_ROOT/gallery_build" \\\n  --gallery-out "$OUT_ROOT/pritom_coco_gallery.npz" \\\n  --metadata-out "$OUT_ROOT/pritom_coco_gallery.json"\n\npython clean_demo_v2/tools/identify_probe_with_gallery.py \\\n  --gallery-npz "$OUT_ROOT/pritom_coco_gallery.npz" \\\n  --video-path "$CLEAN_ROOT/probes/test1probe.mp4" \\\n  --model "$MODEL" \\\n  --threshold -1 \\\n  --margin 0 \\\n  --match-mode max \\\n  --work-root "$WORK_ROOT/probe_test1" \\\n  --output-json "$OUT_ROOT/test1_pritom_coco_closed_result.json"\n\npython - "$OUT_ROOT" <<\'PY\'\nimport json\nimport sys\nfrom pathlib import Path\n\nout_root = Path(sys.argv[1])\nresult_path = out_root / "test1_pritom_coco_closed_result.json"\npayload = json.loads(result_path.read_text())\nsummary_path = out_root / "test1_pritom_coco_closed_summary.txt"\nlines = [\n    "Pritom/Coco closed-set Test1 summary",\n    f"Counts: {payload[\'summary\'][\'counts\']}",\n    f"Processed entries: {payload[\'summary\'][\'processed_entries\']}",\n    f"Skipped entries: {payload[\'summary\'][\'skipped_entries\']}",\n    "",\n    "Per-entry assignments:",\n]\nfor entry in payload["entries"]:\n    if entry["status"] != "ok":\n        lines.append(f"{entry[\'entry_key\']}: skipped {entry[\'status\']}")\n        continue\n    lines.append(\n        f"{entry[\'entry_key\']}: {entry[\'assigned_identity\']} "\n        f"best={entry[\'best_score\']:.4f} second={entry[\'second_identity\']}:{entry[\'second_score\']:.4f}"\n    )\nsummary_path.write_text("\\n".join(lines))\nprint(json.dumps(payload["summary"], indent=2))\nprint(f"[pritom-coco-closed] saved summary: {summary_path}")\nPY\n\npython clean_demo_v2/tools/render_closed_set_annotations.py \\\n  --video-path "$CLEAN_ROOT/probes/test1probe.mp4" \\\n  --tracking-txt "$WORK_ROOT/probe_test1/tracking/test1probe/test1probe.txt" \\\n  --result-json "$OUT_ROOT/test1_pritom_coco_closed_result.json" \\\n  --output-video "$OUT_ROOT/test1_pritom_coco_closed_annotated.mp4" \\\n  --title "Pritom vs Coco closed-set"\n\necho "[pritom-coco-closed] finished"\necho "[pritom-coco-closed] output folder: $OUT_ROOT"\necho "[pritom-coco-closed] result JSON: $OUT_ROOT/test1_pritom_coco_closed_result.json"\necho "[pritom-coco-closed] annotated video: $OUT_ROOT/test1_pritom_coco_closed_annotated.mp4"\n')
path.chmod(0o755)
print('wrote', path)


## 7. Download Gallery And Probe Videos

Gallery Drive folder: https://drive.google.com/drive/folders/19OQwGdyLc-QuAh5JqzLEg58uK7XnGFwH?usp=drive_link

Probe Drive folder: https://drive.google.com/drive/folders/1qSraegJ3hF_RjnIf4Gn1nwZ-2GPoJHIZ?usp=drive_link

For the fixed demo, this cell selects one Pritom gallery, one Coco gallery, and Test1 probe by filename when possible. If the printed selection is wrong, edit the three `*_INDEX` values.

In [ ]:
from pathlib import Path
import shutil

ROOT = Path('/content/All-in-One-Gait')
clean = ROOT / 'clean_demo_v2'
raw_gallery = clean / 'raw_gallery_drive'
raw_probe = clean / 'raw_probe_drive'
(clean / 'gallery').mkdir(parents=True, exist_ok=True)
(clean / 'probes').mkdir(parents=True, exist_ok=True)
raw_gallery.mkdir(parents=True, exist_ok=True)
raw_probe.mkdir(parents=True, exist_ok=True)

GALLERY_URL = 'https://drive.google.com/drive/folders/19OQwGdyLc-QuAh5JqzLEg58uK7XnGFwH?usp=drive_link'
PROBE_URL = 'https://drive.google.com/drive/folders/1qSraegJ3hF_RjnIf4Gn1nwZ-2GPoJHIZ?usp=drive_link'

!gdown --folder "$GALLERY_URL" -O "$raw_gallery" --remaining-ok
!gdown --folder "$PROBE_URL" -O "$raw_probe" --remaining-ok

video_exts = {'.mp4', '.avi', '.mov', '.mkv'}
gallery_videos = sorted([p for p in raw_gallery.rglob('*') if p.suffix.lower() in video_exts])
probe_videos = sorted([p for p in raw_probe.rglob('*') if p.suffix.lower() in video_exts])

print('Gallery videos:')
for i, p in enumerate(gallery_videos):
    print(i, p.name, '->', p)
print('
Probe videos:')
for i, p in enumerate(probe_videos):
    print(i, p.name, '->', p)

if len(gallery_videos) < 2:
    raise RuntimeError('Need at least two gallery videos for this closed-set demo.')
if len(probe_videos) < 1:
    raise RuntimeError('Need at least one probe video.')

def find_index(paths, keywords, fallback):
    keywords = [k.lower() for k in keywords]
    for i, p in enumerate(paths):
        name = p.name.lower()
        if all(k in name for k in keywords):
            return i
    return fallback

PRITOM_INDEX = find_index(gallery_videos, ['pritom'], 0)
COCO_INDEX = find_index(gallery_videos, ['coco'], 1 if PRITOM_INDEX != 1 else 0)
TEST1_INDEX = find_index(probe_videos, ['test1'], 0)

pritom_src = gallery_videos[PRITOM_INDEX]
coco_src = gallery_videos[COCO_INDEX]
probe_src = probe_videos[TEST1_INDEX]

shutil.copy(pritom_src, clean / 'gallery/pritomgallery.mp4')
shutil.copy(coco_src, clean / 'gallery/cocogallery.mp4')
shutil.copy(probe_src, clean / 'probes/test1probe.mp4')

print('
Selected:')
print('pritomgallery:', pritom_src)
print('cocogallery:', coco_src)
print('test1probe:', probe_src)


## 8. Run The Closed-Set Demo

This uses the original demo-style logic: each probe entry is assigned to the most similar gallery identity. No threshold or margin is used in this closed-set mode.

In [ ]:
%cd /content/All-in-One-Gait
!MODEL=grew_gaitbase bash clean_demo_v2/run_pritom_coco_test1_closed.sh


## 9. Inspect Latest Output

This prints the latest output folder, summary, JSON, and annotated video path.

In [ ]:
from pathlib import Path
import json
ROOT = Path('/content/All-in-One-Gait')
outs = sorted((ROOT / 'clean_demo_v2/output').glob('pritom_coco_test1_closed_*'))
if not outs:
    raise RuntimeError('No closed-set output folders found yet.')
latest = outs[-1]
print('latest:', latest)
summary = latest / 'test1_pritom_coco_closed_summary.txt'
result = latest / 'test1_pritom_coco_closed_result.json'
video = latest / 'test1_pritom_coco_closed_annotated.mp4'
if summary.exists():
    print('
SUMMARY
' + summary.read_text())
if result.exists():
    payload = json.loads(result.read_text())
    print('
COUNTS:', payload.get('summary', {}).get('counts'))
print('
annotated video:', video, 'exists:', video.exists())


## 10. Launch Generic Gradio App

This is the temporary deployment interface. Upload any number of gallery videos and one probe video. If there are two or more gallery videos, it runs closed-set best-match recognition. If there is only one gallery video, it falls back to person-vs-unknown threshold mode. Colab will print a public `gradio.live` URL while the cell is running.

In [ ]:
%cd /content/All-in-One-Gait
!python clean_demo_v2/gradio_app.py
